# Unity Environment Walkthrough

Этот ноутбук демонстрирует, как подключить Unity‑среду TensorAeroSpace (Editor или собранный билд) к Python через `mlagents`. Ниже приведён пошаговый сценарий:

1. Установить зависимости (желательно в свежем `venv`/`conda`).
2. Указать путь к Unity‑билду или оставить `None` для подключения к Editor.
3. Создать `UnityEnvironment`, настроить флаги визуализации/серверного режима и обернуть его в `gym`.
4. (Опционально) применить дискретизацию действий через `unity_discrete_env`.
5. Выполнить пару шагов и корректно закрыть соединение.

> ℹ️ Если запускаете в headless/серверном режиме, выставьте переменную `SERVER_MODE=True` (см. ниже) и убедитесь, что Unity‑билд собран с флагом `Run In Background`.


In [ ]:
!pip install mlagents==1.1.0


## 1. Установка зависимостей

Запустите эту ячейку **после** активации виртуального окружения (`python -m venv .venv && source .venv/bin/activate` или `conda create -n tas python=3.10 && conda activate tas`).

- `mlagents` и `mlagents_envs` должны совпадать с версией, на которой собран Unity‑проект (в репозитории используется 1.1.0).
- Для работы с TensorAeroSpace установите также `gym==0.20.0` и `gym-unity==0.28.0` (см. `docs/ru/guide/unity_env.md`).
- При запуске на сервере без дисплея дополнительно поставьте `xvfb` или используйте headless‑билд.


In [ ]:
from pathlib import Path
from typing import Optional

from mlagents_envs.environment import UnityEnvironment
from mlagents_envs.envs.unity_gym_env import UnityToGymWrapper


def get_plane_env(
    env_path: Optional[str] = "",
    server: bool = False,
    worker: int = 0,
    log_dir: str = "/app/logs",
    additional_args: Optional[list[str]] = None,
):
    """Создаёт gym-обёртку для UnityAirplaneEnvironment.

    Args:
        env_path: путь к собранному билду. `None`/"" → подключение к Editor.
        server: `True` для headless-запуска (отключает графику в Unity).
        worker: уникальный ID для параллельных копий (избегает конфликтов портов).
        log_dir: каталог, куда Unity будет писать логи.
        additional_args: дополнительные аргументы для процесса Unity (например, `-logfile`).
    """

    resolved_path = "" if env_path in (None, "") else str(env_path)
    log_dir_path = Path(log_dir)
    log_dir_path.mkdir(parents=True, exist_ok=True)
    unity_args = additional_args or ["-logfile", str(log_dir_path / "unity.log")]

    print(
        f"Connecting to {'Unity Editor' if resolved_path == '' else resolved_path} | "
        f"server={server} | worker_id={worker} | logs={log_dir_path}"
    )

    unity_env = UnityEnvironment(
        resolved_path,
        worker_id=worker,
        no_graphics=server,
        log_folder=str(log_dir_path),
        additional_args=unity_args,
    )

    env = UnityToGymWrapper(unity_env, uint8_visual=True)
    return env


In [ ]:
# ↳ Укажите путь к билду или оставьте None для подключения к редактору
ENV_PATH = None  # Например: Path("/tf/linux_build/build.x86_64")

# Включайте SERVER_MODE=True, если запускаете на сервере без окна
SERVER_MODE = False

# Используйте разные worker_id при параллельных экспериментах
WORKER_ID = 0

# Папка под логи Unity (создаётся автоматически)
LOG_DIR = Path("/app/logs")
ADDITIONAL_ARGS = ["-logfile", str(LOG_DIR / "unity.log")]


## 2. Конфигурация подключения

Заполните переменные ниже под ваш сценарий:

- `ENV_PATH` — абсолютный путь к собранному Unity‑билду (`*.x86_64`, `.app`, `.exe`). Если хотите привязаться к Unity Editor, оставьте `None` и откройте сцену в Play‑режиме.
- `SERVER_MODE` — `True`, если запускаете без окна/на сервере (включает `no_graphics=True`).
- `WORKER_ID` — целое число для параллельных запусков (разные ID предотвращают конфликт портов).
- `LOG_DIR` — каталог, куда Unity будет писать логи (по умолчанию `/app/logs/`).

При первом запуске полезно убедиться, что:
1. В Unity сцене указан `Behavior Parameters → Behavior Name`, совпадающий с ожидаемым в Python.
2. В Build Settings сцена добавлена в список.
3. ML-Agents версии 2.2.1-exp.1 (как в примере) совместимы с установленными Python пакетами.


## 3. Создание `UnityEnvironment` и `gym`-обёртки

Вызов `get_plane_env` ниже:

1. Создаёт папку для логов (если нужно) и передаёт `-logfile` Unity.
2. Автоматически переключается между Unity Editor (когда `ENV_PATH=None`) и standalone‑билдом.
3. Возвращает `UnityToGymWrapper`, совместимую с агентами TensorAeroSpace.

Если соединение не устанавливается:
- Убедитесь, что Editor находится в Play‑режиме (при `ENV_PATH=None`).
- Проверьте, что порт свободен или измените `WORKER_ID`.
- На Linux предоставьте исполняемые права файлу билда (`chmod +x`).


In [ ]:
env = get_plane_env(
    env_path=ENV_PATH,
    server=SERVER_MODE,
    worker=WORKER_ID,
    log_dir=str(LOG_DIR),
    additional_args=ADDITIONAL_ARGS,
)
print("Action space:", env.action_space)
print("Observation space:", env.observation_space)


## 4. Дискретизация действий (опционально)

`unity_discrete_env` преобразует многомерное действие (7 каналов по 3 значения) в одно целое число `0 … 3^7-1`. Это удобно для алгоритмов, ожидающих дискретное пространство действий (DQN, A3C и т.п.).

Если вам нужна непрерывная версия, пропустите этот шаг и работайте напрямую с `env`.


In [ ]:
from tensoraerospace.envs.unity_env import unity_discrete_env

# Преобразуем непрерывное действие (7 каналов) в один дискретный индекс
wrapped_env = unity_discrete_env(env)
print("Discrete action space:", wrapped_env.action_space)


## 5. Smoke-test соединения

Ниже проверяем базовый цикл `reset → step → close`:

1. `reset()` должен вернуть наблюдения без ошибок. Если получаете `UnityEnvironmentException`, проверьте лог (`LOG_DIR`).
2. `step(action)` использует дискретное действие `1` (русифицированный канал элеронов). Замените на `wrapped_env.action_space.sample()` или действие агента.
3. После теста обязательно вызывайте `close()`, иначе Unity‑процесс останется висеть и заблокирует порт.

> ❗ Если Unity сообщает «Display 1 No cameras rendering», откройте сцену и убедитесь, что активная камера привязана к Display 1 (см. чек-лист в `docs/ru/guide/unity_env.md`).


In [ ]:
initial_obs = wrapped_env.reset()
print("Initial observation shape:", getattr(initial_obs, "shape", type(initial_obs)))

sample_action = wrapped_env.action_space.sample()
print("Sample action:", sample_action)

observation, reward, done, info = wrapped_env.step(sample_action)
print(
    f"Reward={reward:.3f} | done={done} | info keys={list(info.keys()) if isinstance(info, dict) else info}"
)

wrapped_env.close()
print("Unity environment closed cleanly ✅")
